# Visao geral do pipeline climatico

Este notebook usa dados ficticios para demonstrar uma preparacao minima antes de qualquer etapa de modelagem. O objetivo e mostrar estrutura, rastreabilidade e padronizacao.

In [ ]:
from pathlib import Path

import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

inmet_path = RAW_DIR / "exemplo_inmet_ficticio.csv"
era5_path = RAW_DIR / "exemplo_era5_ficticio.csv"

## Leitura dos dados ficticios

In [ ]:
inmet = pd.read_csv(inmet_path)
era5 = pd.read_csv(era5_path)

display(inmet.head())
display(era5.head())

## Checagem de colunas

In [ ]:
colunas_inmet = {
    "data_hora",
    "fonte",
    "estacao",
    "temperatura_c",
    "umidade_relativa_pct",
    "precipitacao_mm",
}

colunas_era5 = {
    "data_hora",
    "fonte",
    "ponto_grade",
    "temperatura_2m_k",
    "precipitacao_total_m",
}

assert colunas_inmet.issubset(inmet.columns)
assert colunas_era5.issubset(era5.columns)

print("Estruturas minimas conferidas.")

## Padronizacao de datas

In [ ]:
inmet["data_hora"] = pd.to_datetime(inmet["data_hora"])
era5["data_hora"] = pd.to_datetime(era5["data_hora"])

print(inmet["data_hora"].dtype)
print(era5["data_hora"].dtype)

## Comparacao de resolucao temporal

In [ ]:
freq_inmet = inmet["data_hora"].sort_values().diff().dropna().mode().iloc[0]
freq_era5 = era5["data_hora"].sort_values().diff().dropna().mode().iloc[0]

pd.DataFrame(
    {
        "fonte": ["INMET_EXEMPLO", "ERA5_EXEMPLO"],
        "resolucao_inferida": [freq_inmet, freq_era5],
    }
)

## Padronizacao de unidades e base processada ficticia

A temperatura do exemplo ERA5 e convertida de Kelvin para Celsius. A precipitacao do exemplo ERA5 e convertida de metros para milimetros.

In [ ]:
inmet_padronizado = inmet.rename(columns={"temperatura_c": "temperatura_inmet_c"})

era5_padronizado = era5.copy()
era5_padronizado["temperatura_era5_c"] = era5_padronizado["temperatura_2m_k"] - 273.15
era5_padronizado["precipitacao_era5_mm"] = era5_padronizado["precipitacao_total_m"] * 1000
era5_padronizado = era5_padronizado[["data_hora", "temperatura_era5_c", "precipitacao_era5_mm"]]

base_processada = inmet_padronizado.merge(era5_padronizado, on="data_hora", how="inner")
base_processada = base_processada[
    [
        "data_hora",
        "temperatura_inmet_c",
        "umidade_relativa_pct",
        "precipitacao_mm",
        "temperatura_era5_c",
        "precipitacao_era5_mm",
    ]
]

base_processada

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
saida = PROCESSED_DIR / "base_climatica_ficticia_processada.csv"
base_processada.to_csv(saida, index=False)
print(f"Base ficticia salva em: {saida}")